# 03 – Befolkningstäthet och folkmängd

**Syfte:** Göra om SCB:s befolkningstäthet och folkmängd per kommun 2000–2025 till en analysfärdig tabell. Tabellen får SKR:s kommungrupper A/B/C och samma urval av kommuner som fertilitetstabellen.

| | |
|---|---|
| **Källa** | SCB, Statistikdatabasen: befolkningstäthet och folkmängd per kommun. SKR: *Kommungruppsindelning 2023* |
| **Rådata** | `data/raw/befolkningstathet.csv` (oförändrad, commit `d7b709c`), `data/raw/Kommungruppsindelning-2023-SKR.xlsx` |
| **Hjälpfil** | `data/processed/exkluderade_kommuner.csv` (från `02`) |
| **Resultat** | `data/processed/befolkningstathet.csv` |
| **Beroenden** | Kör `02_FertilitetPerKommun` först. Paketet `openpyxl` behövs för att läsa SKR:s Excelfil. |

**Om rådatat:** Det allra första täthetsutdraget i projektet (commit `4bd1fa5`) var en bred tabell från 1991 och saknade folkmängd. Det ersattes samma dag av utdraget ovan, som har både täthet och folkmängd i långt format. Det senare är det teamet har arbetat med, och därför används det här.

### Beslut som genomförs
1. **Knivstas nollor 2000–2001 blir `NaN`.** Kommunen fanns inte då, så 0 invånare är ett fel och inget mätvärde.
2. **Kommungrupp A/B/C läggs till** från SKR:s egen arbetsbok, kopplad på kommunkod.
3. **Samma kommuner utesluts som i `02`**, alltså de med minst 75 % bortfall i fertilitetsdata.

## 1. Inställningar

Sökvägar som i övriga notebooks. Cellen kontrollerar också att `openpyxl` finns. Paketet behövs för att läsa Excelfilen från SKR. Saknas det stoppar cellen med en tydlig instruktion, i stället för att ge ett svårläst fel längre ner.

In [1]:
from pathlib import Path

import pandas as pd

# Notebooken körs normalt från mappen Assignment.
# Körs den från projektroten letar vi upp Assignment ändå.
BAS = Path.cwd()
if not (BAS / "data" / "raw").is_dir() and (BAS / "Assignment").is_dir():
    BAS = BAS / "Assignment"

RAW = BAS / "data" / "raw"
PROCESSED = BAS / "data" / "processed"
PROCESSED.mkdir(parents=True, exist_ok=True)

assert RAW.is_dir(), "Hittar inte data/raw. Öppna notebooken från mappen Assignment."

try:
    import openpyxl  # noqa: F401 – används av pd.read_excel
except ModuleNotFoundError as fel:
    raise ModuleNotFoundError(
        "openpyxl saknas. Installera det i notebookens miljö: pip install openpyxl"
    ) from fel

## 2. Läs in rådata

Filen är kommaseparerad och kodad i **UTF-8 med BOM**, alltså `encoding="utf-8-sig"`.

Den gamla notebooken läste filen med `encoding="cp1252"`. Då blev första kolumnen `ï»¿Kommunkod` i stället för `Kommunkod`, och *Upplands Väsby* blev *Upplands VÃ¤sby*. Felet märks först när tabellerna ska kopplas ihop.

`Kommunkod` läses som **text**. Annars tolkas `0114` som talet 114, och koden matchar inte längre andra tabeller.

In [2]:
tathet_ra = pd.read_csv(RAW / "befolkningstathet.csv", encoding="utf-8-sig", dtype={"Kommunkod": str})

display(tathet_ra.head())
print("Form:", tathet_ra.shape)
print(tathet_ra.dtypes)

,Kommunkod,Kommun,Kon,År,Befolkningstäthet (invånare per kvkm),Folkmängd
0,0114,Upplands Väsby,1+2,2000,498.6,37576.0
1,0114,Upplands Väsby,1+2,2001,497.9,37524.0
2,0114,Upplands Väsby,1+2,2002,496.8,37444.0
3,0114,Upplands Väsby,1+2,2003,496.2,37397.0
4,0114,Upplands Väsby,1+2,2004,497.8,37517.0


Form: (7540, 6)
Kommunkod                                    str
Kommun                                       str
Kon                                          str
År                                         int64
Befolkningstäthet (invånare per kvkm)    float64
Folkmängd                                float64
dtype: object


## 3. Kontrollera strukturen

Vi förväntar oss **290 kommuner × 26 år = 7 540 rader**, fyrsiffriga kommunkoder, inga dubbletter och inga tomma fält.

Kolumnen `Kon` har bara värdet `1+2`, alltså båda könen. Tätheten gäller hela befolkningen. Kolumnen innehåller ingen information och tas bort.

In [3]:
TATHET = "Befolkningstäthet (invånare per kvkm)"

assert list(tathet_ra.columns) == ["Kommunkod", "Kommun", "Kon", "År", TATHET, "Folkmängd"]
assert tathet_ra["Kommunkod"].str.fullmatch(r"\d{4}").all()
assert tathet_ra["Kommunkod"].nunique() == 290
assert tathet_ra["År"].min() == 2000 and tathet_ra["År"].max() == 2025
assert len(tathet_ra) == 290 * 26
assert not tathet_ra.duplicated(["Kommunkod", "År"]).any()
assert tathet_ra["Kon"].unique().tolist() == ["1+2"]
assert tathet_ra.notna().all().all()

print("Täthet:", tathet_ra[TATHET].min(), "–", tathet_ra[TATHET].max(), "inv/km²")
print("Folkmängd:", int(tathet_ra["Folkmängd"].min()), "–", int(tathet_ra["Folkmängd"].max()))

Täthet: 0.0 – 6529.2 inv/km²
Folkmängd: 0 – 999239


## 4. Knivsta: nollor är fel, inte mätvärden

Filen innehåller två rader där både täthet och folkmängd är **0**: Knivsta 2000 och 2001. Knivsta bröts ut ur Uppsala och blev egen kommun 2003. Då fanns kommunen helt enkelt inte. Värdet ska vara *saknas*, inte noll. Med en nolla kvar skulle Knivsta räknas som Sveriges glesast befolkade kommun i en klassindelning.

Knivsta har däremot ett **riktigt värde för 2002**, och det är korrekt. Uppsalas folkmängd sjunker med cirka 11 400 personer mellan 2001 och 2002, alltså ungefär Knivstas storlek. Siffrorna för 2002 redovisas redan enligt den nya indelningen. Därför behålls 2002.

Samma sak innebär ett **brott i Uppsalas serie**: fram till 2001 ingår Knivsta i Uppsala. Det bör man känna till om Uppsala studeras över tid.

In [4]:
nollor = tathet_ra.loc[(tathet_ra[TATHET] == 0) | (tathet_ra["Folkmängd"] == 0), ["Kommun", "År", TATHET, "Folkmängd"]]
display(nollor)
assert nollor["Kommun"].eq("Knivsta").all() and nollor["År"].tolist() == [2000, 2001]

tathet = tathet_ra.copy()
tathet.loc[nollor.index, [TATHET, "Folkmängd"]] = float("nan")

uppsala = tathet_ra.loc[tathet_ra["Kommun"] == "Uppsala"].set_index("År")["Folkmängd"]
knivsta_2002 = tathet_ra.loc[(tathet_ra["Kommun"] == "Knivsta") & (tathet_ra["År"] == 2002), "Folkmängd"].iloc[0]
print(f"Uppsala 2001 → 2002: {uppsala[2002] - uppsala[2001]:+,.0f} invånare. Knivsta 2002: {knivsta_2002:,.0f}")

,Kommun,År,Befolkningstäthet (invånare per kvkm),Folkmängd
728,Knivsta,2000,0.0,0.0
729,Knivsta,2001,0.0,0.0


Uppsala 2001 → 2002: -11,437 invånare. Knivsta 2002: 12,586


## 5. Känd begränsning: tätheten är avrundad till en decimal

SCB redovisar tätheten med **en decimal**. För de glesaste kommunerna försvinner då all utveckling över tid. Jokkmokk står till exempel på 0,3 invånare/km² alla år, trots att folkmängden sjönk med över 20 %. Tabellen visar kommunerna under 1 invånare/km².

Det går inte att räkna tillbaka en exaktare täthet utan en separat källa för kommunernas landareal. Därför åtgärdas det inte här. Konsekvensen för analysen:

- För att **rangordna och klassindela** kommuner fungerar tätheten bra. De här kommunerna hamnar i den lägsta klassen oavsett.
- För **förändring över tid** i glesa kommuner ska folkmängden användas, inte tätheten.

In [5]:
glesa_kommuner = tathet.loc[tathet[TATHET] < 1, "Kommun"].unique()
urval = tathet.loc[tathet["Kommun"].isin(glesa_kommuner)]

glesa = urval.groupby("Kommun").agg(tathet_min=(TATHET, "min"), tathet_max=(TATHET, "max"))
folkmangd_per_ar = urval.pivot(index="Kommun", columns="År", values="Folkmängd")
glesa["folkmangd_2000"] = folkmangd_per_ar[2000]
glesa["folkmangd_2025"] = folkmangd_per_ar[2025]
glesa["folkmangd_forandring_procent"] = ((glesa["folkmangd_2025"] / glesa["folkmangd_2000"] - 1) * 100).round(0)
display(glesa)

,tathet_min,tathet_max,folkmangd_2000,folkmangd_2025,folkmangd_forandring_procent
Kommun,,,,,
Arjeplog,0.2,0.3,3384.0,2570.0,-24.0
Dorotea,0.8,1.2,3353.0,2241.0,-33.0
Härjedalen,0.9,1.0,11415.0,10117.0,-11.0
Jokkmokk,0.2,0.3,6019.0,4695.0,-22.0
Pajala,0.7,0.9,7480.0,5706.0,-24.0
Sorsele,0.3,0.4,3195.0,2362.0,-26.0
Storuman,0.8,0.9,6934.0,5543.0,-20.0
Vilhelmina,0.8,1.0,7918.0,6172.0,-22.0
Åsele,0.6,0.8,3624.0,2683.0,-26.0


## 6. Kommungrupper enligt SKR 2023

Kommungrupperna hämtas direkt från **SKR:s arbetsbok** (blad *Bilaga1 Lista alla kommuner*). Det är originalkällan. Den tidigare filen `A,B,C.csv` byggdes från samma blad, och grupperna är identiska för alla 290 kommuner.

Arbetsboken har en rubrikrad ovanför tabellen, så `header=1` används. Kolumnnamnet `" Huvudgrupp"` har ett inledande mellanslag i originalet, och därför trimmas alla kolumnnamn. Kommunkoden läses som text.

**Viktigt om tolkningen:** A/B/C är **inte** en indelning i liten, mellan och stor kommun. SKR delar in efter *struktur*, främst närhet till större städer och pendling:

| Grupp | Innebörd | Kommuner |
|---|---|---|
| A | Storstäder och storstadsnära kommuner | 46 |
| B | Större städer och kommuner nära större stad | 110 |
| C | Mindre städer/tätorter och landsbygdskommuner | 134 |

Därför är till exempel Uppsala (cirka 250 000 invånare) grupp **B**, och grupperna överlappar kraftigt i storlek, vilket tabellen nedan visar. Om frågan gäller storlek ska folkmängd eller täthet användas.

De nio undergrupperna (A1–C9) sparas också för mer detaljerade analyser. Observera att **2023 års** indelning används för hela perioden 2000–2025. Det beskriver kommunerna utifrån hur de ser ut i dag, inte hur de klassades vid varje tidpunkt.

In [6]:
skr = pd.read_excel(
    RAW / "Kommungruppsindelning-2023-SKR.xlsx",
    sheet_name="Bilaga1 Lista alla kommuner",
    header=1,
    dtype={"Kommunkod": str},
)
skr.columns = skr.columns.str.strip()
skr = skr.dropna(subset=["Kommunkod"])
skr = skr.rename(columns={
    "Gruppkod": "kommungrupp_kod",
    "Kommunnamn": "Kommun_SKR",
    "Huvudgrupp": "kommungrupp_namn",
    "Kommungrupp 2023": "kommungrupp_undergrupp",
})
skr["kommungrupp"] = skr["kommungrupp_kod"].str[0]

assert len(skr) == 290 and not skr.duplicated("Kommunkod").any()
assert skr["Kommunkod"].str.fullmatch(r"\d{4}").all()
assert skr["kommungrupp"].value_counts().to_dict() == {"C": 134, "B": 110, "A": 46}

tathet = tathet.merge(
    skr[["Kommunkod", "Kommun_SKR", "kommungrupp", "kommungrupp_kod", "kommungrupp_namn", "kommungrupp_undergrupp"]],
    on="Kommunkod", how="left", validate="many_to_one",
)
assert tathet["kommungrupp"].notna().all(), "Någon kommunkod saknas i SKR:s lista."
assert (tathet["Kommun"] == tathet["Kommun_SKR"]).all(), "Kommunnamnen skiljer sig mellan SCB och SKR."
tathet = tathet.drop(columns="Kommun_SKR")

storlek = tathet.loc[tathet["År"] == 2025].groupby("kommungrupp")["Folkmängd"].agg(["count", "min", "median", "max"])
display(storlek.astype(int))

,count,min,median,max
kommungrupp,,,,
A,46,9784,43306,999239
B,110,2327,17168,249726
C,134,2241,12706,75405


## 7. Uteslut samma kommuner som i fertilitetsdata

Kommuner med minst 75 % bortfall i fertilitetsdata är redan uteslutna i `02_FertilitetPerKommun`. Samma kommuner tas bort här. Då innehåller alla bearbetade filer på kommunnivå exakt samma kommuner, och en senare sammankoppling tappar inga rader i tysthet.

Listan läses från `exkluderade_kommuner.csv` i stället för att skrivas in för hand. Om gränsen ändras i `02` följer den här notebooken automatiskt med.

In [7]:
exkl_fil = PROCESSED / "exkluderade_kommuner.csv"
assert exkl_fil.exists(), "Kör 02_FertilitetPerKommun först – den skapar listan över uteslutna kommuner."

exkluderade = pd.read_csv(exkl_fil, encoding="utf-8-sig", dtype={"Kommunkod": str})
display(exkluderade[["Kommunkod", "Kommun", "n_ar_med_data"]].merge(
    skr[["Kommunkod", "kommungrupp"]], on="Kommunkod"))

fore = tathet["Kommunkod"].nunique()
tathet = tathet.loc[~tathet["Kommunkod"].isin(exkluderade["Kommunkod"])]
print(f"Uteslöt {fore - tathet['Kommunkod'].nunique()} kommuner. Kvar: {tathet['Kommunkod'].nunique()}.")

,Kommunkod,Kommun,n_ar_med_data,kommungrupp
0,2403,Bjurholm,0,B
1,2425,Dorotea,2,C
2,2422,Sorsele,6,C
3,2513,Överkalix,8,C
4,2463,Åsele,12,C
5,2506,Arjeplog,12,C


Uteslöt 6 kommuner. Kvar: 284.


## 8. Städa kolumner och datatyper

- Kolumnnamnen görs om till korta ASCII-namn (`ar`, `befolkningstathet`, `folkmangd`).
- `Kon` tas bort, eftersom den är konstant.
- Folkmängden är ett antal personer och sparas som heltal. SCB-filen har den som decimaltal (`37576.0`). Typen `Int64` används eftersom den tillåter saknade värden, vilket behövs för Knivsta 2000–2001.

In [8]:
tathet = (
    tathet.rename(columns={"År": "ar", TATHET: "befolkningstathet", "Folkmängd": "folkmangd"})
    [["Kommunkod", "Kommun", "ar", "befolkningstathet", "folkmangd",
      "kommungrupp", "kommungrupp_kod", "kommungrupp_namn", "kommungrupp_undergrupp"]]
    .sort_values(["Kommunkod", "ar"])
    .reset_index(drop=True)
)
assert (tathet["folkmangd"].dropna() % 1 == 0).all()
tathet["folkmangd"] = tathet["folkmangd"].round().astype("Int64")

display(tathet.head())
print(tathet.dtypes)

,Kommunkod,Kommun,ar,befolkningstathet,folkmangd,kommungrupp,kommungrupp_kod,kommungrupp_namn,kommungrupp_undergrupp
0,0114,Upplands Väsby,2000,498.6,37576,A,A2,Storstäder och storstadsnära kommuner,Pendlingskommun nära storstad
1,0114,Upplands Väsby,2001,497.9,37524,A,A2,Storstäder och storstadsnära kommuner,Pendlingskommun nära storstad
2,0114,Upplands Väsby,2002,496.8,37444,A,A2,Storstäder och storstadsnära kommuner,Pendlingskommun nära storstad
3,0114,Upplands Väsby,2003,496.2,37397,A,A2,Storstäder och storstadsnära kommuner,Pendlingskommun nära storstad
4,0114,Upplands Väsby,2004,497.8,37517,A,A2,Storstäder och storstadsnära kommuner,Pendlingskommun nära storstad


Kommunkod                     str
Kommun                        str
ar                          int64
befolkningstathet         float64
folkmangd                   Int64
kommungrupp                   str
kommungrupp_kod               str
kommungrupp_namn              str
kommungrupp_undergrupp        str
dtype: object


## 9. Slutkontroll

Vi kontrollerar att:

- varje kommun har 26 år och att nyckeln `Kommunkod` + `ar` är unik
- de enda saknade värdena är Knivsta 2000–2001
- kommunerna är **exakt desamma** som i `fertilitet_kommun.csv` från `02`, så att en senare sammankoppling blir ett-till-ett.

In [9]:
assert not tathet.duplicated(["Kommunkod", "ar"]).any()
assert (tathet.groupby("Kommunkod").size() == 26).all()

saknas = tathet.loc[tathet["befolkningstathet"].isna(), ["Kommun", "ar"]]
assert saknas["Kommun"].eq("Knivsta").all() and saknas["ar"].tolist() == [2000, 2001]

fert = pd.read_csv(PROCESSED / "fertilitet_kommun.csv", encoding="utf-8-sig", dtype={"Kommunkod": str})
assert set(fert["Kommunkod"]) == set(tathet["Kommunkod"]), "Kommunurvalet skiljer sig från 02."

print(f"{tathet['Kommunkod'].nunique()} kommuner × 26 år = {len(tathet)} rader")
print("Kommuner per grupp:", tathet.groupby("kommungrupp")["Kommunkod"].nunique().to_dict())
print("Samma kommuner som i fertilitet_kommun.csv: ja")

284 kommuner × 26 år = 7384 rader
Kommuner per grupp: {'A': 46, 'B': 109, 'C': 129}
Samma kommuner som i fertilitet_kommun.csv: ja


## 10. Spara

Kommunkoden sparas som text med inledande nolla. Läs därför in filen med `dtype={"Kommunkod": str}`.

In [10]:
utfil = PROCESSED / "befolkningstathet.csv"
tathet.to_csv(utfil, index=False, encoding="utf-8-sig")

print(f"Sparade {len(tathet)} rader till {utfil.relative_to(BAS)}")

Sparade 7384 rader till data/processed/befolkningstathet.csv
